# Naïve Bayesian Classifier

This notebook demonstrates the implementation of a Naïve Bayesian classifier for a sample training data set stored as a .CSV file, and computes the accuracy of the classifier using test data.

## Importing Required Libraries

We'll start by importing the necessary libraries for implementing the Naïve Bayes classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# For reproducibility
np.random.seed(42)

## Loading and Preparing the Data

Let's first generate a sample dataset for demonstration. In a real scenario, you would load your data from a CSV file.

In [ ]:
# Create a sample dataset for email classification (spam vs. not spam)
# Features: word counts for select keywords

# Generate some synthetic data
np.random.seed(42)
n_samples = 200

# Feature matrix: counts of specific words in emails
# Words like 'free', 'money', 'urgent', 'meeting', 'report'
data = {
    'free_count': np.random.poisson(lam=0.5, size=n_samples),
    'money_count': np.random.poisson(lam=0.3, size=n_samples),
    'urgent_count': np.random.poisson(lam=0.4, size=n_samples),
    'meeting_count': np.random.poisson(lam=0.8, size=n_samples),
    'report_count': np.random.poisson(lam=0.6, size=n_samples)
}

# Create DataFrame
df = pd.DataFrame(data)

# Generate target variable (spam or not): influenced by the word counts
# Higher counts of 'free', 'money', 'urgent' increase spam likelihood
# Higher counts of 'meeting', 'report' decrease spam likelihood
prob_spam = 1 / (1 + np.exp(-(df['free_count'] * 0.7 + 
                            df['money_count'] * 0.8 + 
                            df['urgent_count'] * 0.6 - 
                            df['meeting_count'] * 0.5 - 
                            df['report_count'] * 0.4 - 0.5)))

df['is_spam'] = (np.random.random(n_samples) < prob_spam).astype(int)

# Save the dataset to a CSV file
df.to_csv('email_spam_data.csv', index=False)
print("Sample data saved to 'email_spam_data.csv'")

# Display the first few rows of the dataset
df.head()

In [ ]:
# Alternative: Load data from a CSV file
# Uncomment the line below to load your own CSV file
# df = pd.read_csv('email_spam_data.csv')
# df.head()

## Data Exploration and Visualization

Let's explore and visualize the dataset to better understand it.

In [ ]:
# Basic statistics of the dataset
print("Dataset shape:", df.shape)
print("\nBasic statistics:")
print(df.describe())

# Class distribution
print("\nClass distribution:")
print(df['is_spam'].value_counts())
print(f"Percentage of spam emails: {df['is_spam'].mean() * 100:.2f}%")

In [ ]:
# Visualizing the distribution of each feature by class
plt.figure(figsize=(15, 10))

features = ['free_count', 'money_count', 'urgent_count', 'meeting_count', 'report_count']
for i, feature in enumerate(features):
    plt.subplot(2, 3, i+1)
    sns.boxplot(x='is_spam', y=feature, data=df)
    plt.title(f'{feature} vs is_spam')
    plt.xlabel('Is Spam (1=Yes, 0=No)')
    
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## Data Preparation for Modeling

Let's split our data into training and testing sets.

In [ ]:
# Split features and target
X = df.drop('is_spam', axis=1)
y = df['is_spam']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## Building and Training the Naïve Bayes Classifier

Now we'll implement different types of Naïve Bayes classifiers and compare their performance.

### 1. Gaussian Naïve Bayes

Suitable for continuous data with a Gaussian distribution.

In [ ]:
# Initialize Gaussian Naïve Bayes classifier
gnb = GaussianNB()

# Train the model
gnb.fit(X_train, y_train)

# Make predictions
gnb_pred = gnb.predict(X_test)
gnb_prob = gnb.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class

# Calculate accuracy
gnb_accuracy = accuracy_score(y_test, gnb_pred)
print(f"Gaussian Naïve Bayes Accuracy: {gnb_accuracy:.4f}")

# Classification report
print("\nClassification Report (Gaussian NB):")
print(classification_report(y_test, gnb_pred))

### 2. Multinomial Naïve Bayes

Suitable for discrete data like word counts in text classification.

In [ ]:
# Initialize Multinomial Naïve Bayes classifier
mnb = MultinomialNB()

# Train the model
mnb.fit(X_train, y_train)

# Make predictions
mnb_pred = mnb.predict(X_test)
mnb_prob = mnb.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class

# Calculate accuracy
mnb_accuracy = accuracy_score(y_test, mnb_pred)
print(f"Multinomial Naïve Bayes Accuracy: {mnb_accuracy:.4f}")

# Classification report
print("\nClassification Report (Multinomial NB):")
print(classification_report(y_test, mnb_pred))

### 3. Bernoulli Naïve Bayes

Suitable for binary/boolean features.

In [ ]:
# Initialize Bernoulli Naïve Bayes classifier
bnb = BernoulliNB()

# Create binary features for Bernoulli NB (presence or absence of words)
X_train_binary = (X_train > 0).astype(int)
X_test_binary = (X_test > 0).astype(int)

# Train the model
bnb.fit(X_train_binary, y_train)

# Make predictions
bnb_pred = bnb.predict(X_test_binary)
bnb_prob = bnb.predict_proba(X_test_binary)[:, 1]  # Probability estimates for the positive class

# Calculate accuracy
bnb_accuracy = accuracy_score(y_test, bnb_pred)
print(f"Bernoulli Naïve Bayes Accuracy: {bnb_accuracy:.4f}")

# Classification report
print("\nClassification Report (Bernoulli NB):")
print(classification_report(y_test, bnb_pred))

## Comparing the Naïve Bayes Classifiers

Let's visualize and compare the performance of the three types of Naïve Bayes classifiers.

In [ ]:
# Comparing accuracy
models = ['Gaussian NB', 'Multinomial NB', 'Bernoulli NB']
accuracies = [gnb_accuracy, mnb_accuracy, bnb_accuracy]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=accuracies)
plt.title('Accuracy Comparison of Naïve Bayes Classifiers')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
for i, acc in enumerate(accuracies):
    plt.text(i, acc + 0.01, f'{acc:.4f}', ha='center')
plt.show()

In [ ]:
# ROC curves for all classifiers
plt.figure(figsize=(10, 8))

# Gaussian NB ROC
fpr_gnb, tpr_gnb, _ = roc_curve(y_test, gnb_prob)
roc_auc_gnb = auc(fpr_gnb, tpr_gnb)
plt.plot(fpr_gnb, tpr_gnb, label=f'Gaussian NB (AUC = {roc_auc_gnb:.3f})')

# Multinomial NB ROC
fpr_mnb, tpr_mnb, _ = roc_curve(y_test, mnb_prob)
roc_auc_mnb = auc(fpr_mnb, tpr_mnb)
plt.plot(fpr_mnb, tpr_mnb, label=f'Multinomial NB (AUC = {roc_auc_mnb:.3f})')

# Bernoulli NB ROC
fpr_bnb, tpr_bnb, _ = roc_curve(y_test, bnb_prob)
roc_auc_bnb = auc(fpr_bnb, tpr_bnb)
plt.plot(fpr_bnb, tpr_bnb, label=f'Bernoulli NB (AUC = {roc_auc_bnb:.3f})')

# Random classifier line
plt.plot([0, 1], [0, 1], 'k--', label='Random')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Naïve Bayes Classifiers')
plt.legend(loc='best')
plt.show()

In [ ]:
# Confusion matrices for all classifiers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gaussian NB confusion matrix
cm_gnb = confusion_matrix(y_test, gnb_pred)
sns.heatmap(cm_gnb, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
axes[0].set_title('Gaussian NB')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Multinomial NB confusion matrix
cm_mnb = confusion_matrix(y_test, mnb_pred)
sns.heatmap(cm_mnb, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
axes[1].set_title('Multinomial NB')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

# Bernoulli NB confusion matrix
cm_bnb = confusion_matrix(y_test, bnb_pred)
sns.heatmap(cm_bnb, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
axes[2].set_title('Bernoulli NB')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.suptitle('Confusion Matrices for Naïve Bayes Classifiers', y=1.05)
plt.show()

## Feature Importance in Naïve Bayes

Let's examine how each feature influences the classification decision.

In [ ]:
# For Multinomial NB, we can look at feature log probabilities
if hasattr(mnb, 'feature_log_prob_'):
    feature_importance = np.exp(mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0])
    
    # Create DataFrame for feature importance
    feature_imp_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': feature_importance
    })
    
    # Sort by importance
    feature_imp_df = feature_imp_df.sort_values('Importance', ascending=False)
    
    # Display feature importance
    print("Feature importance (likelihood ratio for spam vs. not spam):")
    print(feature_imp_df)
    
    # Visualize feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_imp_df)
    plt.title('Feature Importance for Spam Classification')
    plt.xlabel('Importance (Likelihood Ratio)')
    plt.tight_layout()
    plt.show()

## Classifying New Emails

Let's use our best Naïve Bayes classifier to classify new emails.

In [ ]:
# Function to classify new emails
def classify_email(free_count, money_count, urgent_count, meeting_count, report_count, model_type='best'):
    # Create a DataFrame for the new email
    new_email = pd.DataFrame({
        'free_count': [free_count],
        'money_count': [money_count],
        'urgent_count': [urgent_count],
        'meeting_count': [meeting_count],
        'report_count': [report_count]
    })
    
    # Select the model to use
    if model_type == 'gaussian':
        model = gnb
        new_email_processed = new_email
    elif model_type == 'multinomial':
        model = mnb
        new_email_processed = new_email
    elif model_type == 'bernoulli':
        model = bnb
        new_email_processed = (new_email > 0).astype(int)
    else:  # Use the model with highest accuracy
        accuracies = [gnb_accuracy, mnb_accuracy, bnb_accuracy]
        best_idx = np.argmax(accuracies)
        
        if best_idx == 0:  # Gaussian
            model = gnb
            new_email_processed = new_email
            model_name = 'Gaussian NB'
        elif best_idx == 1:  # Multinomial
            model = mnb
            new_email_processed = new_email
            model_name = 'Multinomial NB'
        else:  # Bernoulli
            model = bnb
            new_email_processed = (new_email > 0).astype(int)
            model_name = 'Bernoulli NB'
    
    # Make prediction
    prediction = model.predict(new_email_processed)[0]
    probability = model.predict_proba(new_email_processed)[0, 1]  # Probability of spam
    
    if model_type == 'best':
        print(f"Using {model_name} (best model)")
        
    return prediction, probability

# Example: Classify a new email
new_email_features = {
    'free_count': 2,
    'money_count': 3,
    'urgent_count': 1,
    'meeting_count': 0,
    'report_count': 0
}

prediction, probability = classify_email(
    new_email_features['free_count'],
    new_email_features['money_count'],
    new_email_features['urgent_count'],
    new_email_features['meeting_count'],
    new_email_features['report_count']
)

print("\nNew Email Features:")
for feature, count in new_email_features.items():
    print(f"{feature}: {count}")

print(f"\nPrediction: {'Spam' if prediction == 1 else 'Not Spam'}")
print(f"Probability of being spam: {probability:.4f}")

In [ ]:
# Try another example with different feature values
new_email_features = {
    'free_count': 0,
    'money_count': 0,
    'urgent_count': 0,
    'meeting_count': 2,
    'report_count': 3
}

prediction, probability = classify_email(
    new_email_features['free_count'],
    new_email_features['money_count'],
    new_email_features['urgent_count'],
    new_email_features['meeting_count'],
    new_email_features['report_count']
)

print("\nNew Email Features:")
for feature, count in new_email_features.items():
    print(f"{feature}: {count}")

print(f"\nPrediction: {'Spam' if prediction == 1 else 'Not Spam'}")
print(f"Probability of being spam: {probability:.4f}")

## Conclusion

In this notebook, we have implemented and demonstrated the Naïve Bayesian classifier for an email spam classification problem. We showed how to:

1. Prepare and explore the dataset
2. Implement three different types of Naïve Bayes classifiers: Gaussian, Multinomial, and Bernoulli
3. Compare the performance of these classifiers using accuracy, ROC curves, and confusion matrices
4. Examine feature importance in Naïve Bayes classification
5. Use the classifier to predict whether new emails are spam or not

Naïve Bayes classifiers are particularly effective for text classification tasks due to their simplicity, speed, and ability to handle high-dimensional data with relatively small training datasets. They make a strong independence assumption, which is often not realistic, but they still perform surprisingly well in many real-world applications.